# Description de la classe TensorProv et de ses fonctions

La classe `TensorProv` est conçue pour capturer la provenance des opérations effectuées sur des DataFrames. Elle utilise des tenseurs creux (sparse tensors) pour suivre et représenter les relations entre les enregistrements d'origine et les enregistrements transformés dans différentes opérations comme la réduction, l'augmentation, les jointures et les ajouts. Cette approche permet une gestion efficace de la traçabilité des données, même pour de grands ensembles de données.

## Objectif principal
- Fournir une manière structurée de capturer et représenter la provenance des données.
- Supporter deux méthodes de capture de provenance :
  - **`record_id`** : Attribue un identifiant unique à chaque ligne.
  - **`hashing`** : Génère un identifiant basé sur le hachage des valeurs d'une ligne.

## Description des fonctions principales

### `__init__`
- **Objectif** : Initialise l'instance de la classe et définit la méthode utilisée pour capturer la provenance (`record_id` ou `hashing`).
- **Attributs** :
  - `method` : Méthode de capture de provenance.
  - `record_id_counter` : Compteur pour générer des identifiants uniques lorsque `record_id` est utilisé.

### `_generate_hash`
- **Objectif** : Générer un identifiant unique basé sur le hachage des valeurs d'une ligne.
- **Utilisation** : Employé lorsque la méthode de capture de provenance est `hashing`.

### `_add_record_ids`
- **Objectif** : Ajouter une colonne d'identifiants uniques (`_record_id` ou `_hash_id`) à chaque ligne du DataFrame.
- **Détails** :
  - Si la méthode est `record_id`, les identifiants sont incrémentiels.
  - Si la méthode est `hashing`, les identifiants sont calculés à partir des valeurs de chaque ligne.

### `horizontal_data_reduction`
- **Objectif** : Filtrer les lignes d'un DataFrame selon une condition (réduction horizontale).
- **Entrées** :
  - `df` : Le DataFrame d'entrée.
  - `condition` : Une condition utilisée pour filtrer les lignes (ex. : `"Gender == 'F'"`).
- **Sorties** :
  - Le DataFrame filtré.
  - Un tenseur creux qui relie les lignes filtrées aux lignes d'origine.

### `vertical_data_reduction`
- **Objectif** : Réduire les colonnes d'un DataFrame selon une liste donnée (réduction verticale).
- **Entrées** :
  - `df` : Le DataFrame d'entrée.
  - `columns_to_keep` : Liste des colonnes à conserver.
- **Sorties** :
  - Le DataFrame réduit aux colonnes sélectionnées.
  - Un tenseur creux traçant les colonnes conservées.

### `horizontal_data_augmentation`
- **Objectif** : Ajouter des exemples synthétiques à un DataFrame (augmentation horizontale).
- **Détails** :
  - Sélectionne aléatoirement des lignes pour générer de nouvelles instances.
  - Ajoute du bruit aux colonnes numériques pour varier les données synthétiques.
  - Capture la provenance des nouvelles lignes.
- **Entrées** :
  - `df` : Le DataFrame d'entrée.
  - `n_samples` : Nombre d'instances synthétiques à générer.
- **Sorties** :
  - Le DataFrame enrichi avec les nouvelles instances.
  - Un tenseur creux reliant les lignes d'origine et synthétiques.

### `vertical_data_augmentation`
- **Objectif** : Ajouter de nouvelles colonnes avec des valeurs calculées ou constantes (augmentation verticale).
- **Détails** :
  - Les nouvelles colonnes peuvent contenir des valeurs fixes ou générées dynamiquement à l'aide de fonctions.
- **Entrées** :
  - `df` : Le DataFrame d'entrée.
  - `new_columns` : Dictionnaire où les clés sont les noms des nouvelles colonnes et les valeurs des constantes ou des fonctions génératrices.
- **Sorties** :
  - Le DataFrame enrichi avec les nouvelles colonnes.
  - Un tenseur creux traçant les colonnes ajoutées.

### `join`
- **Objectif** : Effectuer une jointure entre deux DataFrames et capturer la provenance des lignes résultantes.
- **Entrées** :
  - `df_left` : Premier DataFrame (gauche).
  - `df_right` : Deuxième DataFrame (droite).
  - `on` : Colonne(s) utilisée(s) pour la jointure.
  - `how` : Type de jointure (`inner`, `left`, etc.).
- **Sorties** :
  - Le DataFrame résultant de la jointure.
  - Deux tenseurs creux représentant la contribution des lignes des DataFrames gauche et droit au DataFrame final.


## Conclusion
La classe `TensorProv` est un outil puissant pour capturer et analyser la provenance des données lors de diverses transformations. Elle garantit une traçabilité claire et efficace, essentielle pour des tâches de gestion et d'audit de données. Chaque fonction est conçue pour être modulaire et extensible, permettant une intégration facile dans des workflows plus larges.



In [61]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import time

class TensorProv:
    def __init__(self, method='record_id'):
        self.method = method
        self.record_id_counter = 0

    def _generate_hash(self, row, max_value):
        """Génère un hash unique pour une ligne et le normalise."""
        row_str = ','.join(map(str, row))
        return abs(hash(row_str)) % max_value

    def _add_record_ids(self, df):
        """Ajoute un identifiant ou un hash selon la méthode choisie."""
        df = df.copy()  # Copie pour éviter des modifications accidentelles
        if self.method == 'record_id':
            if '_record_id' not in df.columns:
                df['_record_id'] = range(self.record_id_counter, self.record_id_counter + len(df))
                self.record_id_counter += len(df)
                print("Colonne '_record_id' ajoutée avec succès.")
        elif self.method == 'hashing':
            if '_hash_id' not in df.columns:
                df['_hash_id'] = df.apply(lambda row: self._generate_hash(row, len(df)), axis=1)
                print("Colonne '_hash_id' ajoutée avec succès.")
        return df

    def horizontal_data_reduction(self, df, condition):
        """Filtre les lignes selon une condition (réduction horizontale)."""
        df = self._add_record_ids(df)
        df_out = df.query(condition).reset_index(drop=True)

        # Début du chronométrage
        start_time = time.time()

        ids = df_out['_record_id'].values if self.method == 'record_id' else df_out['_hash_id'].values
        rows = np.arange(len(df_out))

        # Vérification des indices
        print(f"IDs générés (max: {len(df) - 1}): {ids.max()}")

        tensor = csr_matrix((np.ones(len(rows)), (rows, ids)), shape=(len(df_out), len(df)))

        # Temps écoulé pour constituer le tenseur
        elapsed_time = time.time() - start_time
        print(f"Tenseur constitué en {elapsed_time:.4f} secondes.")

        return df_out.drop(columns=['_record_id', '_hash_id'], errors='ignore'), tensor

    def vertical_data_reduction(self, df, columns_to_keep):
        """Réduit les colonnes selon une liste donnée (réduction verticale)."""
        df = self._add_record_ids(df)
        print(f"Colonnes actuelles après ajout des identifiants : {df.columns.tolist()}")

        id_column = '_record_id' if self.method == 'record_id' else '_hash_id'
        if id_column not in df.columns:
            raise KeyError(f"'{id_column}' n'est pas dans les colonnes du DataFrame.")

        df_out = df[columns_to_keep + [id_column]]

        # Début du chronométrage
        start_time = time.time()

        tensor = csr_matrix((np.ones(len(df)), (np.arange(len(df)), np.arange(len(df)))), shape=(len(df), len(df)))

        # Temps écoulé pour constituer le tenseur
        elapsed_time = time.time() - start_time
        print(f"Tenseur constitué en {elapsed_time:.4f} secondes.")

        return df_out.drop(columns=[id_column], errors='ignore'), tensor

    def horizontal_data_augmentation(self, df, n_samples):
        """Ajoute des exemples synthétiques (augmentation horizontale)."""
        df = self._add_record_ids(df)
        indices = np.random.choice(len(df), size=n_samples, replace=True)
        synthetic_data = df.iloc[indices].copy()
        numeric_cols = synthetic_data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != '_record_id':
                synthetic_data[col] += np.random.normal(0, 0.1, size=len(synthetic_data))
        df_out = pd.concat([df, synthetic_data]).reset_index(drop=True)

        # Début du chronométrage
        start_time = time.time()

        rows = np.arange(len(df_out))
        cols = np.concatenate((np.arange(len(df)), indices))
        tensor = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df_out), len(df)))

        # Temps écoulé pour constituer le tenseur
        elapsed_time = time.time() - start_time
        print(f"Tenseur constitué en {elapsed_time:.4f} secondes.")

        return df_out.drop(columns=['_record_id', '_hash_id'], errors='ignore'), tensor

    def vertical_data_augmentation(self, df, new_columns):
        """Ajoute de nouvelles colonnes au DataFrame (augmentation verticale)."""
        df = self._add_record_ids(df)

        # Ajout des nouvelles colonnes
        for col_name, col_func in new_columns.items():
            df[col_name] = col_func(df)

        # Début du chronométrage
        start_time = time.time()

        tensor = csr_matrix((np.ones(len(df)), (np.arange(len(df)), np.arange(len(df)))), shape=(len(df), len(df)))

        # Temps écoulé pour constituer le tenseur
        elapsed_time = time.time() - start_time
        print(f"Tenseur constitué en {elapsed_time:.4f} secondes.")

        return df.drop(columns=['_record_id', '_hash_id'], errors='ignore'), tensor

    def join(self, df_left, df_right, on, how='inner'):
        """Jointure de deux DataFrames avec suivi de provenance."""
        df_left = self._add_record_ids(df_left)
        df_right = self._add_record_ids(df_right)
        df_out = pd.merge(df_left, df_right, on=on, how=how)

        # Sélectionner les colonnes d'identifiant appropriées
        id_left = '_record_id' if self.method == 'record_id' else '_hash_id'
        id_right = '_record_id' if self.method == 'record_id' else '_hash_id'

        left_ids = df_out[f'{id_left}_x'].values
        right_ids = df_out[f'{id_right}_y'].values

        # Réindexer les IDs pour qu'ils correspondent aux dimensions des DataFrames
        left_ids = np.clip(left_ids, 0, len(df_left) - 1)
        right_ids = np.clip(right_ids, 0, len(df_right) - 1)

        # Début du chronométrage
        start_time = time.time()

        left_tensor = csr_matrix((np.ones(len(left_ids)), (np.arange(len(left_ids)), left_ids)),
                                 shape=(len(df_out), len(df_left)))
        right_tensor = csr_matrix((np.ones(len(right_ids)), (np.arange(len(right_ids)), right_ids)),
                                  shape=(len(df_out), len(df_right)))

        # Temps écoulé pour constituer les tenseurs
        elapsed_time = time.time() - start_time
        print(f"Tenseurs constitués en {elapsed_time:.4f} secondes.")

        return df_out.drop(columns=[f'{id_left}_x', f'{id_right}_y', '_hash_id_x', '_hash_id_y'], errors='ignore'), (left_tensor, right_tensor)


class TensorProvTests:
    def __init__(self, data_path):
        self.data_path = data_path
        self.tp = None
        self.customer_data = None

    def load_data(self):
        """Charge les données client."""
        self.customer_data = pd.read_csv(self.data_path)
        print(f"Dataset chargé : {self.customer_data.shape} lignes, {self.customer_data.shape[1]} colonnes.")

    def initialize_tensorprov(self, method):
        """Initialise TensorProv avec la méthode spécifiée."""
        self.tp = TensorProv(method=method)
        print(f"Instance TensorProv initialisée avec la méthode '{method}'.")

    def display_tensor(self, tensor, name):
        """Affiche le tenseur sous forme dense."""
        print(f"\n{name} (forme : {tensor.shape})")
        print("Aperçu des valeurs du tenseur (dense) :")
        print(tensor.toarray())

    def test_operation(self, method, operation_name, operation_func):
        """Teste une opération et affiche les résultats."""
        print(f"\n=== Test {operation_name} avec la méthode '{method}' ===")
        result = operation_func()
        result_df = result[0]
        result_tensor = result[1]

        print(f"Taille du DataFrame résultant : {result_df.shape}")
        print(result_df.head())

        # Gérer les cas où plusieurs tenseurs sont retournés
        if isinstance(result_tensor, tuple):
            print(f"Les tenseurs retournés pour '{operation_name}' :")
            for i, tensor in enumerate(result_tensor):
                print(f"Tenseur {i + 1} (forme : {tensor.shape}):")
                self.display_tensor(tensor, f"Tenseur {i + 1}")
        else:
            print(f"Taille du tenseur : {result_tensor.shape}")
            self.display_tensor(result_tensor, f"Tenseur obtenu ({operation_name})")

    def run_all_tests(self):
        """Exécute tous les tests pour les deux méthodes."""
        for method in ['record_id', 'hashing']:
            self.initialize_tensorprov(method)
            self.test_operation(method, "Horizontal Data Reduction",
                                lambda: self.tp.horizontal_data_reduction(self.customer_data.copy(), "Age > 50"))
            self.test_operation(method, "Vertical Data Reduction",
                                lambda: self.tp.vertical_data_reduction(self.customer_data.copy(), ["CustomerID", "Age", "City"]))
            self.test_operation(method, "Horizontal Data Augmentation",
                                lambda: self.tp.horizontal_data_augmentation(self.customer_data.copy(), n_samples=10))
            self.test_operation(method, "Vertical Data Augmentation",
                                lambda: self.tp.vertical_data_augmentation(self.customer_data.copy(), {
                                    "NewFeature": lambda df: np.random.rand(len(df))
                                }))
            self.test_operation(method, "Join",
                                lambda: self.tp.join(self.customer_data.copy(),
                                                     pd.DataFrame({
                                                         "CustomerID": np.random.randint(1, 31, size=10),
                                                         "PurchaseAmount": np.random.uniform(10, 100, size=10)
                                                     }),
                                                     on="CustomerID", how="inner"))


# Chemin vers le fichier client
data_path = "/Users/zgherari/Documents/Projets-Master-IA/Quality/customer_data_large.csv"

# Exécution des tests
tester = TensorProvTests(data_path)
tester.load_data()
tester.run_all_tests()


Dataset chargé : (1000000, 8) lignes, 8 colonnes.
Instance TensorProv initialisée avec la méthode 'record_id'.

=== Test Horizontal Data Reduction avec la méthode 'record_id' ===
Colonne '_record_id' ajoutée avec succès.
IDs générés (max: 999999): 999998
Tenseur constitué en 0.0026 secondes.
Taille du DataFrame résultant : (365103, 8)
   CustomerID     Name  Age      Gender         City           SignupDate  \
0           2      Eve   64  Non-Binary      Houston  2016-03-07 12:00:00   
1           3  Charlie   67      Female     New York  2012-10-16 12:00:00   
2           5      Eve   53  Non-Binary      Phoenix  2020-08-12 12:00:00   
3           7  Charlie   67      Female      Chicago  2014-07-20 12:00:00   
4          10      Eve   62  Non-Binary  Los Angeles  2012-03-11 12:00:00   

   AnnualSpend  LoyaltyPoints  
0      3869.02           2768  
1     15772.53           1778  
2      6396.46           1181  
3     13644.40           3460  
4      5611.51           1721  
Taille d